In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as mlt
import seaborn as sns

In [4]:
df=pd.read_csv('f1data.csv')

In [5]:
df.head

<bound method NDFrame.head of         speed_kmh  wing_angle_deg  drs_active  downforce_n  drag_n  \
0          199.25            7.08           0       577.87   96.66   
1          351.94           22.57           0      5617.09  378.93   
2          293.98           28.97           0      4945.14  300.02   
3          258.64           27.93           0      3702.69  227.46   
4          141.34           30.12           1      1184.59   53.24   
...           ...             ...         ...          ...     ...   
149995     160.88            7.78           1       414.19   47.55   
149996     123.76           23.77           0       729.44   47.96   
149997     151.97           23.80           1      1101.16   54.26   
149998     161.50            6.43           0       345.11   63.18   
149999     200.83           26.12           0      2098.18  132.26   

        stability_index  
0                100.00  
1                 44.15  
2                 77.74  
3                100.00  

In [7]:
df.isnull().sum()

speed_kmh          0
wing_angle_deg     0
drs_active         0
downforce_n        0
drag_n             0
stability_index    0
dtype: int64

In [10]:
df.drop(['stability_index'],axis=1,inplace=True)

In [11]:
df.head()

,speed_kmh,wing_angle_deg,drs_active,downforce_n,drag_n
0,199.25,7.08,0,577.87,96.66
1,351.94,22.57,0,5617.09,378.93
2,293.98,28.97,0,4945.14,300.02
3,258.64,27.93,0,3702.69,227.46
4,141.34,30.12,1,1184.59,53.24


In [12]:
df.isna().sum()

speed_kmh         0
wing_angle_deg    0
drs_active        0
downforce_n       0
drag_n            0
dtype: int64

In [13]:
from sklearn.preprocessing import StandardScaler

In [14]:
scaler = StandardScaler()
df[['speed_kmh','wing_angle_deg','downforce_n','drag_n']]=scaler.fit_transform(df[['speed_kmh','wing_angle_deg','downforce_n','drag_n']])

In [15]:
df.head()

,speed_kmh,wing_angle_deg,drs_active,downforce_n,drag_n
0,-0.435738,-1.494272,0,-0.984473,-0.652766
1,1.561139,0.294314,0,1.747802,1.997907
2,0.803139,1.033304,0,1.383470,1.256898
3,0.340963,0.913219,0,0.709811,0.575519
4,-1.193084,1.166092,1,-0.655508,-1.060504


In [16]:
df.to_csv('processed_data.csv',index=False)

In [17]:
df_new=pd.read_csv('processed_data.csv')

In [18]:
df_new

,speed_kmh,wing_angle_deg,drs_active,downforce_n,drag_n
0,-0.435738,-1.494272,0,-0.984473,-0.652766
1,1.561139,0.294314,0,1.747802,1.997907
2,0.803139,1.033304,0,1.383470,1.256898
3,0.340963,0.913219,0,0.709811,0.575519
4,-1.193084,1.166092,1,-0.655508,-1.060504
...,...,...,...,...,...
149995,-0.937540,-1.413445,1,-1.073221,-1.113937
149996,-1.422995,0.432875,0,-0.902292,-1.110087
149997,-1.054065,0.436339,1,-0.700744,-1.050926
149998,-0.929432,-1.569326,0,-1.110676,-0.967162


In [22]:
def check_data(df):
    for col in df.columns:
        print(f"\nColumn: {col}")
        
        if df[col].nunique() < 10:
            print("Categorical distribution (%):")
            print(df[col].value_counts(normalize=True) * 100)
        else:
            print("Continuous summary:")
            print(df[col].describe())
check_data(df_new)


Column: speed_kmh
Continuous summary:
count    1.500000e+05
mean    -3.022175e-17
std      1.000003e+00
min     -1.733727e+00
25%     -8.682267e-01
50%      3.943846e-03
75%      8.668290e-01
max      1.731937e+00
Name: speed_kmh, dtype: float64

Column: wing_angle_deg
Continuous summary:
count    1.500000e+05
mean     1.080972e-16
std      1.000003e+00
min     -1.734444e+00
25%     -8.661309e-01
50%      1.027720e-03
75%      8.635676e-01
max      1.729572e+00
Name: wing_angle_deg, dtype: float64

Column: drs_active
Categorical distribution (%):
drs_active
0    70.102667
1    29.897333
Name: proportion, dtype: float64

Column: downforce_n
Continuous summary:
count    1.500000e+05
mean    -1.241081e-16
std      1.000003e+00
min     -1.240772e+00
25%     -7.936073e-01
50%     -2.997181e-01
75%      5.598024e-01
max      3.570507e+00
Name: downforce_n, dtype: float64

Column: drag_n
Continuous summary:
count    1.500000e+05
mean     1.216449e-16
std      1.000003e+00
min     -1.390958e+

In [34]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor

In [35]:
X=df_new[['speed_kmh','wing_angle_deg','drs_active','downforce_n']]
y=df_new['drag_n']

In [37]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
#initialise models
models ={
    'Linear Regression':LinearRegression(),
    'Lasso':Lasso(random_state=42),
    'Ridge':Ridge(random_state=42),
    
    'Random Forest Regressor':RandomForestRegressor(random_state=42),
    'Decision Tree':DecisionTreeRegressor(random_state=42),
    'Gradient Boosting':GradientBoostingRegressor(random_state=42),

    'KNN':KNeighborsRegressor()
}
for name, model in models.items():
    try:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        print(f"\nModel Name: {name}")
        print(f"R2 Score: {r2_score(y_test, y_pred):.3f}")
        print(f"MAE: {mean_absolute_error(y_test, y_pred):.3f}")
        print(f"MSE: {mean_squared_error(y_test, y_pred):.3f}")

    except Exception as e:
        print(f"\nModel Name: {name} FAILED")
        print(f"Error: {e}")


Model Name: Linear Regression
R2 Score: 0.976
MAE: 0.122
MSE: 0.024

Model Name: Lasso
R2 Score: -0.000
MAE: 0.833
MSE: 0.994

Model Name: Ridge
R2 Score: 0.976
MAE: 0.122
MSE: 0.024

Model Name: Random Forest Regressor
R2 Score: 1.000
MAE: 0.001
MSE: 0.000

Model Name: Decision Tree
R2 Score: 1.000
MAE: 0.004
MSE: 0.000

Model Name: Gradient Boosting
R2 Score: 0.999
MAE: 0.019
MSE: 0.001

Model Name: KNN
R2 Score: 1.000
MAE: 0.003
MSE: 0.000
